# Downloading ALL PubChem locally

## 00. Setup

In [1]:
import requests
import re
from bs4 import BeautifulSoup


### Understand the PubChem BioAssay FTP Folder Structure

PubChem stores all BioAssay files here:

**Descriptions** (XML): https://ftp.ncbi.nlm.nih.gov/pubchem/Bioassay/CSV/Description/

**Data** (CSV assay results): https://ftp.ncbi.nlm.nih.gov/pubchem/Bioassay/CSV/Data/

Inside each folder files look like:

Each zip file corresponds to a range of AIDs. Example:

`0000001_0001000.zip`  → contains assays with AID 1 to 1000

In `Description/`, there are files such as:

Each `*.descr.xml` file follows the official PubChem BioAssay XML schema and may contain detailed metadata describing the assay. Depending on the assay, the XML can include:

Always present (schema-required)
- **Assay Name**: `<PC-AssayDescription_name>`
- **Assay Description / Protocol text**: `<PC-AssayDescription_description>` (may include protocol steps, summary, conditions, etc.)
- **Depositor / Source Information**: `<PC-AssayDescription_aid-source>`


Present when deposited by submitter (schema-optional)
- **Targets** (proteins, genes, taxonomy IDs):` <PC-AssayDescription_target>`
- **Comments / Additional notes**: `<PC-AssayDescription_comment>`
- **References** (PMIDs, DOIs, citation links): `<PC-AssayDescription_xref>`
- **Relations to other assays**: `<PC-AssayDescription_relations>`

In `Data/`, there are files like:

Each .csv contains the assay results:

Columns 1–7:
- `PUBCHEM_RESULT_TAG`
- `PUBCHEM_SID`
- `PUBCHEM_CID`
- `PUBCHEM_ACTIVITY_OUTCOME`
- `PUBCHEM_ACTIVITY_SCORE`
- `PUBCHEM_ACTIVITY_URL`
- `PUBCHEM_ASSAYDATA_COMMENT`

Columns 8+:
- depositor-defined results (IC50, % inhibition, etc.)


### Calculate aprox file size

In [ ]:
def get_total_ftp_size(url):
    print(f"Checking: {url}")
    r = requests.get(url).text

    # Matches lines like: "0000001_0001000.zip     120M"
    sizes = re.findall(r'\s+(\d+(?:\.\d+)?)([KMG])', r)

    total_bytes = 0
    for num, unit in sizes:
        num = float(num)
        if unit == "K": num *= 1e3
        if unit == "M": num *= 1e6
        if unit == "G": num *= 1e9
        total_bytes += num

    return total_bytes

def to_gb(bytes_val):
    return round(bytes_val / 1e9, 3)

desc_url = "https://ftp.ncbi.nlm.nih.gov/pubchem/Bioassay/CSV/Description/"
data_url = "https://ftp.ncbi.nlm.nih.gov/pubchem/Bioassay/CSV/Data/"

desc_total = get_total_ftp_size(desc_url)
data_total = get_total_ftp_size(data_url)

print("\n=== FTP Total Size Summary ===")
print(f"Description/ total: {to_gb(desc_total)} GB")
print(f"Data/ total:         {to_gb(data_total)} GB")
print(f"Combined total:      {to_gb(desc_total + data_total)} GB")

Checking: https://ftp.ncbi.nlm.nih.gov/pubchem/Bioassay/CSV/Description/
Checking: https://ftp.ncbi.nlm.nih.gov/pubchem/Bioassay/CSV/Data/

=== FTP Total Size Summary ===
Description/ total: 3.417 GB
Data/ total:         10.629 GB
Combined total:      14.046 GB


## 1. Get the list of all ZIP files from PubChem

In [ ]:
# Check all the zip files in Description

url = "https://ftp.ncbi.nlm.nih.gov/pubchem/Bioassay/CSV/Description/"
html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser")

zip_files = [a.text for a in soup.find_all("a") if a.text.endswith(".zip")]
zip_files[:10], len(zip_files)

(['0000001_0001000.zip',
  '0001001_0002000.zip',
  '0002001_0003000.zip',
  '0003001_0004000.zip',
  '0004001_0005000.zip',
  '0005001_0006000.zip',
  '0006001_0007000.zip',
  '0007001_0008000.zip',
  '0008001_0009000.zip',
  '0009001_0010000.zip'],
 1792)

In [3]:
# Check all the zip files in Data

url = "https://ftp.ncbi.nlm.nih.gov/pubchem/Bioassay/CSV/Data/"
html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser")

zip_files = [a.text for a in soup.find_all("a") if a.text.endswith(".zip")]
zip_files[:10], len(zip_files)

(['0000001_0001000.zip',
  '0001001_0002000.zip',
  '0002001_0003000.zip',
  '0003001_0004000.zip',
  '0004001_0005000.zip',
  '0005001_0006000.zip',
  '0006001_0007000.zip',
  '0007001_0008000.zip',
  '0008001_0009000.zip',
  '0009001_0010000.zip'],
 1792)